In [1]:
import torch.nn.functional as F
import torch

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [3]:
for w in words:
    if (len(w) == 15):
        print(w)

muhammadibrahim
muhammadmustafa


从前一个字符预测下一个 bigram Language MODEL

- By counting 直接统计 字符a 后接 字符b 的概率

In [4]:
N = torch.zeros((27,27), dtype = torch.int32) 

In [5]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars) }
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [6]:
# 统计 ch1 后是 ch2 的数量
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [7]:
P = (N+1).float() # 加1 model smoothing，防止有概率为0的参数让log likelihood 为负无穷
P /= P.sum(1, keepdim = True)

Broadcasting semantics

In [8]:
g = torch.Generator().manual_seed(2147483647)
out = []

for i in range(10):
    name = ''
    ix = 0
    while True: 
        # p = N[ix].float()
        # p = p/ N[ix].sum()
        p = P[ix]
        # 按每个元素代表概率权重从索引里抽样，返回抽样到的索引 
        # 生成的为张量，用 item() 取值
        ix = torch.multinomial(p, 1, replacement = True,  generator = g).item()
        ch = itos[ix]
        name += ch
        if (ch == '.'):
            break
    out.append(name)
            
print(''.join(out))

cexze.momasurailezitynn.konimittain.llayn.ka.da.staiyaubrtthrigotai.moliellavo.ke.teda.


衡量 quality of the model

**GOAL** : 最大化 likelihood
- 相当于最大化 log likelihood
- 相当于最小化 negative log likelihood
- 相当于最小化 average negative log likelihood

log(a * b * c) = log(a) + log(b) + log(c)

In [9]:
logLikelihood = 0
n = 0
for w in ['andrejq']:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        prob = P[ix1, ix2]
        logp = torch.log(prob)
        logLikelihood += logp
        n+=1
        # print(f'{ch1} {ch2}: {prob.item():.3f} {logp:.3f}')

nll = -logLikelihood;
print(f'{nll}')
print(f'{nll / n}')

27.867216110229492
3.4834020137786865


In [20]:
# create the tranning set of bigrams
xs,ys = [], [] # xs input , ys 期望输出

for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        # print(ch1, ch2)
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs) # input
ys = torch.tensor(ys) # label


In [13]:
W = torch.randn((27, 27), generator = g, requires_grad = True) # random initialize 27 weights

In [14]:
# forward pass
xenc = F.one_hot(xs, num_classes = 27).float() # input to the network, one-hot encoding
logits =  xenc @ W # log-counts
# softmax
counts = logits.exp() # equivalent N，看起来像 counts 
probs = counts / counts.sum(1, keepdim = True) # 

`probs[0]` 输入 . 的情况下 neural network 输出下一个字符的概率 

In [15]:
probs[torch.arange(5), ys] # network 输出正确label的概率

tensor([0.0212, 0.0102, 0.0081, 0.0233, 0.0226], grad_fn=<IndexBackward0>)

In [16]:
loss = -probs[torch.arange(5), ys].log().mean()
loss # loss Func

tensor(4.1589, grad_fn=<NegBackward0>)

In [17]:
# backard pass
W.grad = None
loss.backward()

**Neural Net** 

In [22]:
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)
xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

# initialize the 'network'
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

number of examples:  228146


In [23]:
# gradient descent （梯度下降
for k in range(1):
  
  # forward pass
  xenc = F.one_hot(xs, num_classes=27).float() # input to the network: one-hot encoding
  logits = xenc @ W # predict log-counts
  counts = logits.exp() # counts, equivalent to N
  probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
  loss = -probs[torch.arange(num), ys].log().mean() + 0.01*(W**2).mean()
  print(loss.item())
  
  # backward pass
  W.grad = None # set to zero the gradient
  loss.backward()
  
  # update
  W.data += -50 * W.grad

3.7686190605163574


In [24]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

texzmkloglquszipczktxhkmpmzistttwinmlgdukzka.
zr.
rocxtpucjwtsc.
gmtokmxczisqytxugkwpt.
dajkkluydjmscdgu.
